<h1>Part 4 - Experiment Tracking</h1>

# Experiment Tracking and Model Management with MLFlow

There are many ways to use the MLFlow Tracking API. For simple local uses, the best is to leave the data management to MLFlow and let it store runs, metrics, models and artifacts locally. For more advanced usage, all of this information can be stored in databases. You can find the detailed on MLFlow's documentation [here](https://mlflow.org/docs/latest/tracking.html#scenario-1-mlflow-on-localhost).

## Exploring MLFlow

MLflow setup:
* Tracking server: no
* Backend store: local filesystem
* Artifacts store: local filesystem

The experiments can be explored locally by launching the MLflow UI.

Run `uv sync` at the repository root and select the project's Python kernel. This notebook uses the course's pinned MLflow 2.7.1, including its model-stage API. Run the introduction notebook's download cell first to obtain the three taxi parquet files. Tracking data and registered artifacts stay in this lesson's ignored `mlruns/` directory. Rerunning the notebook creates new runs and model versions without deleting previous experiments.

Let's print the tracking server URI, where the experiments and runs are going to be logged. We observe it refers to a local path.

In [ ]:
from pathlib import Path

import mlflow

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file())
LESSON_FOLDER = REPO_ROOT / "lessons" / "01-model-and-experiment-management"
TRACKING_FOLDER = LESSON_FOLDER / "mlruns"
mlflow.set_tracking_uri(TRACKING_FOLDER.as_uri())
mlflow.set_registry_uri(mlflow.get_tracking_uri())

print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

After this initialization, we can connect create a client to connect to the API and see what experiments are present.

By refering to mlflow's [documentation](https://mlflow.org/docs/latest/python_api/mlflow.client.html), create a client and display a list of the available experiments using the search_experiments function. This function could prove useful later to programatically explore experiments (rather than in the UI)

In [ ]:
from mlflow import MlflowClient

client = MlflowClient()
client.search_experiments()

We see that there is a default experiment for which the runs are stored locally in the mlruns folder.

### Creating an experiment and logging a new run

An experiment is a logical entity regrouping the logs of multiple attempts at solving a same problem, called runs. \
We will now work with the classic sklearn dataset iris. Our goal here is to manage to classify the different iris species. To track our models performance, we will log every attempt as a "run" and create a new experiment "iris-experiment-1" to regroup them.

Lookup the mlflow.run and mlflow.start_run functions [here](https://mlflow.org/docs/latest/python_api/mlflow.html?highlight=start_run#mlflow.start_run) to find out how to manage runs.
Explore [this part](https://mlflow.org/docs/latest/python_api/mlflow.html) to learn more about the log_params, log_metrics and log_artifact functions. Find out how to log sklearn models [here](https://mlflow.org/docs/latest/python_api/mlflow.sklearn.html])

Complete the following in order to log the parameters, interesting metrics and the model.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from mlflow.models import infer_signature

import pandas as pd

mlflow.set_experiment("iris-experiment-1")

X, y = load_iris(return_X_y=True)
X_iris_train, X_iris_valid, y_iris_train, y_iris_valid = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
iris_runs = []
for regularization in (0.01, 0.1, 1.0, 10.0):
    with mlflow.start_run(run_name=f"logistic-regression-C-{regularization}") as run:
        params = {"C": regularization, "random_state": 42, "max_iter": 1000}
        mlflow.set_tags({"dataset": "iris", "model_type": "LogisticRegression"})
        mlflow.log_params({**params, "validation_fraction": 0.2, "split_seed": 42})
        model = LogisticRegression(**params).fit(X_iris_train, y_iris_train)
        y_pred = model.predict(X_iris_valid)
        metrics = {
            "train_accuracy": accuracy_score(y_iris_train, model.predict(X_iris_train)),
            "validation_accuracy": accuracy_score(y_iris_valid, y_pred),
            "validation_f1_macro": f1_score(y_iris_valid, y_pred, average="macro"),
        }
        mlflow.log_metrics(metrics)
        mlflow.log_dict({"feature_names": load_iris().feature_names}, "feature_names.json")
        mlflow.sklearn.log_model(model, "models", signature=infer_signature(X_iris_train, model.predict(X_iris_train)))
        iris_runs.append({"run_id": run.info.run_id, "C": regularization, **metrics})
iris_comparison = pd.DataFrame(iris_runs).sort_values(["validation_accuracy", "C"], ascending=[False, True])
run_id = iris_comparison.iloc[0]["run_id"]
iris_comparison

In [ ]:
client = MlflowClient()
experiments = client.search_experiments()
experiments

The four runs above compare different regularization strengths using a stratified validation split, not training accuracy. The best validation accuracy selects the registered run, with smaller C breaking ties. This validation score is used for selection, so it is not an independent final test score.

To inspect runs, paste the command printed below into a separate terminal. The explicit backend URI ensures the UI uses the same local store regardless of the terminal's working directory.

In [ ]:
print(f"uv run mlflow ui --backend-store-uri '{mlflow.get_tracking_uri()}' --host 127.0.0.1 --port 5002")

Open http://127.0.0.1:5002 after launching the command. Running the UI outside the notebook keeps Run All non-blocking. Stop the UI with Ctrl+C when finished.

### Interacting with the model registry

If you are satisfied with the last run's model, you can transform the logged model into a registered model. It will be logged in the Model Registry, which makes it easier to use in production and manage versions.

In [ ]:
# We already have our run id from above. Let's use it to register the model

result = mlflow.register_model(f"runs:/{run_id}/models", "iris_lr_model")
registered_iris = mlflow.sklearn.load_model(f"models:/iris_lr_model/{result.version}")
assert (registered_iris.predict(X_iris_valid) == mlflow.sklearn.load_model(f"runs:/{run_id}/models").predict(X_iris_valid)).all()
print(f"Registered Iris version {result.version} from run {run_id}")

# Use Case

Now we will get back to our taxi rides use case: 

In [ ]:
from typing import List

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

DATA_FOLDER = REPO_ROOT / "data"
train_path = f"{DATA_FOLDER}/yellow_tripdata_2021-01.parquet"
test_path = f"{DATA_FOLDER}/yellow_tripdata_2021-02.parquet"
predict_path = f"{DATA_FOLDER}/yellow_tripdata_2021-03.parquet"
for path in (train_path, test_path, predict_path):
    if not Path(path).is_file():
        raise FileNotFoundError(f"Missing {path}. Run the introduction notebook's data download cell first.")

## 1 - Load data

In [ ]:
def load_data(path: str) -> pd.DataFrame:
    """Load taxi records from a parquet file."""
    return pd.read_parquet(path)


train_df = load_data(train_path)
train_df.head()

## 2 - Prepare the data

Let's prepare the data to make it Machine Learning ready. \
For this, we need to clean it, compute the target (what we want to predict), and compute some features to help the model understand the data better.

### 2-1 Compute the target

We want to predict a taxi trip duration in minutes. Let's compute it as a difference between the drop-off time and the pick-up time for each trip.

In [ ]:
def compute_target(
    df: pd.DataFrame,
    pickup_column: str = "tpep_pickup_datetime",
    dropoff_column: str = "tpep_dropoff_datetime",
) -> pd.DataFrame:
    """Compute trip duration in minutes without modifying the input."""
    df = df.copy()
    df["duration"] = df[dropoff_column] - df[pickup_column]
    df["duration"] = df["duration"].dt.total_seconds() / 60
    return df


train_df = compute_target(train_df)

In [ ]:
train_df["duration"].describe()

Let's remove outliers and reduce the scope to trips between 1 minute and 1 hour

In [ ]:
MIN_DURATION = 1
MAX_DURATION = 60


def filter_outliers(df: pd.DataFrame, min_duration: int = 1, max_duration: int = 60) -> pd.DataFrame:
    """Keep trips within the inclusive duration limits."""
    return df.loc[df["duration"].between(min_duration, max_duration)].copy()


train_df = filter_outliers(train_df)

### 2-2 Prepare features

#### 2-2-1 Categorical features

Most machine learning models don't work with categorical features. Because of this, they must be transformed so that the ML model can consume them.

In [ ]:
CATEGORICAL_COLS = ["PULocationID", "DOLocationID", "passenger_count"]


def encode_categorical_cols(df: pd.DataFrame, categorical_cols: List[str] = None) -> pd.DataFrame:
    """Convert categorical IDs to strings, reserving -1 for missing values."""
    if categorical_cols is None:
        categorical_cols = CATEGORICAL_COLS
    df = df.copy()
    df[categorical_cols] = df[categorical_cols].fillna(-1).astype("int")
    df[categorical_cols] = df[categorical_cols].astype("str")
    return df


train_df = encode_categorical_cols(train_df)

In [ ]:
def extract_x_y(
    df: pd.DataFrame,
    categorical_cols: List[str] = None,
    dv: DictVectorizer = None,
    with_target: bool = True,
) -> tuple[csr_matrix, np.ndarray | None, DictVectorizer]:
    """Fit features on training data only; reuse the vectorizer for inference."""
    if categorical_cols is None:
        categorical_cols = CATEGORICAL_COLS
    dicts = df[categorical_cols].to_dict(orient="records")

    if dv is None:
        if not with_target:
            raise ValueError("Inference requires a fitted training vectorizer.")
        dv = DictVectorizer().fit(dicts)
    y = df["duration"].to_numpy() if with_target else None

    x = dv.transform(dicts)
    return x, y, dv


X_train, y_train, dv = extract_x_y(train_df)

## 3 - Train model

We train a basic linear regression model to have a baseline performance

In [ ]:
def train_model(x_train: csr_matrix, y_train: np.ndarray) -> LinearRegression:
    """Train a Linear Regression."""
    lr = LinearRegression()
    lr.fit(x_train, y_train)
    return lr


model = train_model(X_train, y_train)

## 4 - Evaluate model

We evaluate the model on train and test data

### 4-1 On train data

In [ ]:
def predict_duration(input_data: csr_matrix, model: LinearRegression) -> np.ndarray:
    """Predict trip duration in minutes from sparse features."""
    return model.predict(input_data)


def evaluate_model(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Compute root mean squared error in minutes."""
    return root_mean_squared_error(y_true, y_pred)


prediction = predict_duration(X_train, model)
train_me = evaluate_model(y_train, prediction)
train_me

### 4-2 On test data

In [ ]:
test_df = load_data(test_path)

In [ ]:
test_df = compute_target(test_df)
test_df = filter_outliers(test_df, MIN_DURATION, MAX_DURATION)
test_df = encode_categorical_cols(test_df)
X_test, y_test, _ = extract_x_y(test_df, dv=dv)

In [ ]:
y_pred_test = predict_duration(X_test, model)
test_me = evaluate_model(y_test, y_pred_test)
test_me

## 5 - Log Model Parameters to MlFlow

Now that all our development functions are built and tested, let's create a training pipeline and log the training parameters, logs and model to MlFlow.

Create a training flow, log all the important parameters, metrics and model. Try to find what could be important and needs to be logged.

In [ ]:
from tempfile import TemporaryDirectory

import joblib

TAXI_MODEL_NAME = "nyc_taxi_duration_lr"
mlflow.set_experiment("nyc-taxi-duration")


def training_flow(training_path: str, evaluation_path: str) -> dict:
    """Train, evaluate, log and register a reproducible taxi model."""
    with mlflow.start_run(run_name="linear-regression-january-2021") as run:
        mlflow.set_tags({"dataset": "nyc-yellow-taxi", "task": "trip-duration", "lesson": "experiment-tracking"})
        training = encode_categorical_cols(filter_outliers(compute_target(load_data(training_path)), MIN_DURATION, MAX_DURATION))
        evaluation = encode_categorical_cols(
            filter_outliers(compute_target(load_data(evaluation_path)), MIN_DURATION, MAX_DURATION)
        )
        x_training, y_training, vectorizer = extract_x_y(training)
        x_evaluation, y_evaluation, _ = extract_x_y(evaluation, dv=vectorizer)
        regressor = train_model(x_training, y_training)
        metrics = {
            "train_rmse": evaluate_model(y_training, regressor.predict(x_training)),
            "test_rmse": evaluate_model(y_evaluation, regressor.predict(x_evaluation)),
            "baseline_test_rmse": evaluate_model(y_evaluation, np.full(y_evaluation.shape, y_training.mean())),
        }
        mlflow.log_params(
            {
                **regressor.get_params(),
                "train_file": Path(training_path).name,
                "test_file": Path(evaluation_path).name,
                "categorical_columns": ",".join(CATEGORICAL_COLS),
                "missing_category": -1,
                "min_duration": MIN_DURATION,
                "max_duration": MAX_DURATION,
                "train_rows": len(training),
                "test_rows": len(evaluation),
                "feature_count": x_training.shape[1],
            }
        )
        mlflow.log_metrics(metrics)
        mlflow.log_dict(
            {"categorical_columns": CATEGORICAL_COLS, "features": vectorizer.get_feature_names_out().tolist()}, "features.json"
        )
        with TemporaryDirectory() as directory:
            vectorizer_path = Path(directory) / "vectorizer.pkl"
            joblib.dump(vectorizer, vectorizer_path)
            mlflow.log_artifact(str(vectorizer_path), artifact_path="preprocessor")
        mlflow.sklearn.log_model(regressor, "models", signature=infer_signature(x_training, regressor.predict(x_training)))
        version = mlflow.register_model(f"runs:/{run.info.run_id}/models", TAXI_MODEL_NAME)
        return {"run_id": run.info.run_id, "version": version.version, **metrics}


taxi_run = training_flow(train_path, test_path)
taxi_run

If the model is satisfactory, we stage it as production using the appropriate version. This will help us retreiving it for predictions.

Create a mlflow client and use the [mlflow documentation](https://mlflow.org/docs/latest/python_api/mlflow.client.html?highlight=transition_model_version_stage#mlflow.client.MlflowClient.transition_model_version_stage) to stage the appropriate model as being in "production".

In [ ]:
client = MlflowClient()
# This is a local teaching registry, not a deployment to a live service.
# Promote only this run's exact version, and only if it beats the baseline.
assert np.isfinite(taxi_run["test_rmse"])
assert taxi_run["test_rmse"] < taxi_run["baseline_test_rmse"], "Model does not beat the baseline"
production_version = client.transition_model_version_stage(
    name=TAXI_MODEL_NAME, version=taxi_run["version"], stage="Production", archive_existing_versions=True
)
assert production_version.current_stage == "Production"
print(f"Production: {TAXI_MODEL_NAME} version {production_version.version}")

## 6 - Predict

We can now use our model to predict on fresh unseen data and forecast what is going to be the duration of a tawi trip depending on trip characteristics.

In [ ]:
# Resolve the Production stage, then pin its version so model and vectorizer match.
production_versions = client.get_latest_versions(TAXI_MODEL_NAME, stages=["Production"])
assert len(production_versions) == 1
production_version = production_versions[0]
assert production_version.run_id == taxi_run["run_id"]
model_uri = f"models:/{TAXI_MODEL_NAME}/{production_version.version}"
model = mlflow.sklearn.load_model(model_uri)

# Retrieve the fitted preprocessor from the same run, not notebook memory.
with TemporaryDirectory() as directory:
    artifact = mlflow.artifacts.download_artifacts(
        artifact_uri=f"runs:/{production_version.run_id}/preprocessor/vectorizer.pkl", dst_path=directory
    )
    production_dv = joblib.load(artifact)

# March is used for inference only: neither timestamps nor the target are required.
predict_df = encode_categorical_cols(load_data(predict_path)[CATEGORICAL_COLS])
X_pred, target, _ = extract_x_y(predict_df, dv=production_dv, with_target=False)
assert target is None
y_pred = predict_duration(X_pred, model)
assert len(y_pred) == len(predict_df) and np.isfinite(y_pred).all()
reference = mlflow.sklearn.load_model(f"runs:/{taxi_run['run_id']}/models")
np.testing.assert_allclose(y_pred[:100], reference.predict(X_pred[:100]))
print(f"Predicted {len(y_pred):,} March trips using registered version {production_version.version}")
pd.Series(y_pred, name="predicted_duration_minutes").describe()

## 7 - To go further

If you managed to go this far, you can try solving the use case using an other regression model like [XGBoost](https://xgboost.readthedocs.io/en/stable/) for instance.

### Results and lessons learned

- **Iris:** four runs compare C = 0.01, 0.1, 1 and 10. Validation accuracies were 80%, 96.67%, 96.67% and 100%, respectively. The C = 10 run was registered, and loading its registered version reproduced its predictions. These are model-selection scores on only 30 validation examples, not evidence of perfect generalization.
- **Taxi:** the tracked linear regression achieved training RMSE **6.7824 minutes** and February test RMSE **7.3711 minutes**, compared with **9.2953 minutes** for the training-mean baseline. Both train and test data use the same 1–60 minute duration filter. The vectorizer is fitted only on January.
- **Reproducibility:** the run records model parameters, data filenames, duration limits, categorical columns, missing-value encoding, row counts, feature count, metrics and feature names. The fitted vectorizer is saved as an artifact alongside the model's run; a model alone would not preserve the mapping from categories to feature columns.
- **Registry:** the exact taxi model version was moved to Production in the local teaching registry after beating the baseline. In a real service, this simple check would need additional validation and approval; assigning a registry stage does not itself deploy a web service.
- **Inference:** loading the registered model and the vectorizer from the same run produced **1,925,152** finite predictions for March, using only booking-time features. No duration target or dropoff timestamp was used. Predictions matched the original run's model on a checked sample.
- **Limitation:** unconstrained linear regression produced some negative durations (minimum about **-12.77 minutes**) and values above the training duration limit (maximum about **71.17 minutes**). Finite predictions are not necessarily plausible predictions. Before actual deployment, investigate affected routes and unseen categories, and validate a positive-target model or an explicit output policy rather than silently clipping results. March was used for inference, not claimed as an evaluated test set.

The optional extension is to track an alternative regressor in the same experiment and compare it using chronological validation, while keeping a later month untouched for final evaluation. Generated run files and notebook outputs are kept out of Git, as required by the repository's hooks.